# 1. Introduction

Création des variables utilisées par les modèles d'intelligence artificielle.

# 2. Importation des bibliothèques

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# 3. Chargement des données nettoyées

In [3]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

gold = pd.read_csv(PROCESSED_DATA_DIR / "gold_clean.csv")
silver = pd.read_csv(PROCESSED_DATA_DIR / "silver_clean.csv")

display(gold.head())

,Datetime,Close,High,Low,Open,Volume,Ticker
0,2024-05-30 04:00:00+00:00,2333.500000,2335.100098,2332.600098,2332.899902,0,GC=F
1,2024-05-30 05:00:00+00:00,2323.699951,2333.699951,2320.800049,2333.000000,617,GC=F
2,2024-05-30 06:00:00+00:00,2331.699951,2331.699951,2322.800049,2323.399902,1913,GC=F
3,2024-05-30 07:00:00+00:00,2355.699951,2356.899902,2331.399902,2331.399902,45741,GC=F
4,2024-05-30 08:00:00+00:00,2356.300049,2357.300049,2330.600098,2355.800049,13770,GC=F


# 4. Création EMA20

In [11]:
gold["EMA20"] = gold["Close"].ewm(span=20).mean()
silver["EMA20"] = silver["Close"].ewm(span=20).mean()

# 5. Création EMA50

In [14]:
gold["EMA50"] = gold["Close"].ewm(span=50).mean()
silver["EMA50"] = silver["Close"].ewm(span=50).mean()

In [15]:
#6. Création EMA200

In [16]:
gold["EMA200"] = gold["Close"].ewm(span=200).mean()
silver["EMA200"] = silver["Close"].ewm(span=200).mean()

# 7. Création RSI

In [17]:
def calculate_rsi(df, period=14):

    delta = df["Close"].diff()

    gain = delta.where(delta > 0, 0)

    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(period).mean()

    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))

    return rsi

gold["RSI"] = calculate_rsi(gold)

silver["RSI"] = calculate_rsi(silver)

# 8. Création ATR

In [18]:
def calculate_atr(df, period=14):

    high_low = df["High"] - df["Low"]

    high_close = abs(df["High"] - df["Close"].shift())

    low_close = abs(df["Low"] - df["Close"].shift())

    tr = pd.concat(
        [high_low, high_close, low_close],
        axis=1
    ).max(axis=1)

    atr = tr.rolling(period).mean()

    return atr

gold["ATR"] = calculate_atr(gold)

silver["ATR"] = calculate_atr(silver)

# 9. Création de la cible IA

In [19]:
gold["Target"] = (
    gold["Close"].shift(-5) > gold["Close"]
).astype(int)

silver["Target"] = (
    silver["Close"].shift(-5) > silver["Close"]
).astype(int)

# 10. Sauvegarde

In [20]:
FEATURES_DIR = PROJECT_ROOT / "data" / "features"

FEATURES_DIR.mkdir(exist_ok=True)

gold.to_csv(FEATURES_DIR / "gold_features.csv", index=False)

silver.to_csv(FEATURES_DIR / "silver_features.csv", index=False)